In [ ]:
import threading
import time

def keep_alive():
    """
    Prints a keep-alive message every 60 seconds.
    Useful to prevent some platforms from timing out due to inactivity.
    """
    while True:
        time.sleep(60)  # Wait 60 seconds
        print("Keeping session alive...")

# Start keep-alive thread (daemon=True means it won't block main program exit)
alive_thread = threading.Thread(target=keep_alive, daemon=True)
alive_thread.start()

In [ ]:
#Conducting Monte Carlo Simulations to Generate and Analyze Single-Case Graphs

#Import packages
import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import pandas as pd

#This function creates a time series with n points, an autocorrelation of a,
#and a constant of ct

def create_time_series(n, a, ct):

    #Create an empty vector to hold values
    time_series = np.empty((0,))

    #Compute first point (no autocorrelation possible)
    point1 = np.random.normal(size = 1)

    #Add point1 to time series
    time_series = np.hstack((time_series, point1))

    #Repeat the process below for all subsequent points
    for i in range(1, n):

        #Compute autocorrelated point error term
        point = a*time_series[i-1]+np.random.normal(size = 1)

        #Add autcorrelated point to time series
        time_series = np.hstack((time_series, point))

    #Add constant to all points
    time_series = time_series + ct

    #Return the time series
    return(time_series)



In [ ]:
#This function creates data for an alternating treatment graph with an
#autocorrelation of a, a trend of tr (in degrees), a constant of ct, a minimum
#of nb_points in each condition, a standardized mean difference of smd,
#and level of r in the control condition. Alternation is semi-random

def create_ME_data(a, trControl, trAttention, trTangibles, trEscape, trAlone, ct, nb_points, smdAttention, smdTangibles, smdEscape, smdAlone, r, alternation):

  if alternation == 'semi-random':

        #Empty vector for labels
        labels = np.empty((0,))

        #Repeat process for each pair of points
        for i in range(nb_points):

            #Randomly select the order of 5 conditions
            new_var = 5
            conditions = np.random.choice(['Control', 'Attention', 'Tangibles', 'Escape', 'Alone'], 5, replace = False)

            #Add conditions to labels
            labels = np.hstack((labels, conditions))

        #Run code until minimum number of points is reached for all phases
        while (np.sum(labels == 'Control') < nb_points) | (np.sum(labels == 'Attention') < nb_points)| (np.sum(labels == 'Tangibles') < nb_points)| (np.sum(labels == 'Escape') < nb_points)| (np.sum(labels == 'Alone') < nb_points):

            #Randomly select one condition
            condition = np.random.choice(['Control', 'Test', 'Tangibles', 'Escape', 'Alone'], 1)

            #Add condition to labels
            labels = np.hstack((labels, condition))

  #Create times series
  time_series = create_time_series(len(labels), a, ct)
  all_values = np.copy(time_series)

  #Indices for Phase A (Control)
  idxControl = np.where(labels == 'Control')[0]

  #Indices for Attention
  idxAttention = np.where(labels == 'Attention')[0]

  #Indices for Tangibles
  idxTangibles = np.where(labels == 'Tangibles')[0]

  #Indices for Escape
  idxEscape = np.where(labels == 'Escape')[0]

  #Indices for Alone
  idxAlone = np.where(labels == 'Alone')[0]

  #Add smd to values of test conditions
  all_values[idxAttention] += smdAttention
  all_values[idxTangibles] += smdTangibles
  all_values[idxEscape] += smdEscape
  all_values[idxAlone] += smdAlone

  #Identify middle point around which to pivot trend
  middle_point = np.median(range(len(all_values)))

  #Apply trend to all points separately
  #Control
  for j in range(len(idxControl)):
    distance = idxControl[j] - middle_point
    all_values[idxControl[j]] += distance * math.tan(trControl * math.pi / 180)

  #Phase Attention
  for j in range(len(idxAttention)):
        distance = idxAttention[j] - middle_point
        all_values[idxAttention[j]] += distance * math.tan(trAttention * math.pi / 180)

  #Phase Tangibles
  for j in range(len(idxTangibles)):
        distance = idxTangibles[j] - middle_point
        all_values[idxTangibles[j]] += distance * math.tan(trTangibles * math.pi / 180)

  #Phase Escape
  for j in range(len(idxEscape)):
        distance = idxEscape[j] - middle_point
        all_values[idxEscape[j]] += distance * math.tan(trEscape * math.pi / 180)

  #Phase Alone
  for j in range(len(idxAlone)):
        distance = idxAlone[j] - middle_point
        all_values[idxAlone[j]] += distance * math.tan(trAlone * math.pi / 180)

  #Set max and min values, and normalize
  min_value = np.min(all_values)
  max_value = np.max(all_values)
  all_values = (all_values - min_value) / (max_value - min_value) * 10

  #Recalculate smd
  mean_control = np.mean(all_values[idxControl])
  mean_attention = np.mean(all_values[idxAttention])
  mean_tangibles = np.mean(all_values[idxTangibles])
  mean_escape = np.mean(all_values[idxEscape])
  mean_alone = np.mean(all_values[idxAlone])

  current_smd_attention = mean_attention - mean_control
  current_smd_tangibles = mean_tangibles - mean_control
  current_smd_escape = mean_escape - mean_control
  current_smd_alone = mean_alone - mean_control

  #Adjust values to maintain smd
  #Attention
  if current_smd_attention != smdAttention:
      adjustment = smdAttention - current_smd_attention
      all_values[idxAttention] += adjustment / 2
      all_values[idxControl] -= adjustment / 2

  #Tangibles
  if current_smd_tangibles != smdTangibles:
      adjustment = smdTangibles - current_smd_tangibles
      all_values[idxTangibles] += adjustment / 2
      all_values[idxControl] -= adjustment / 2

  #Escape
  if current_smd_escape != smdEscape:
      adjustment = smdEscape - current_smd_escape
      all_values[idxEscape] += adjustment / 2
      all_values[idxControl] -= adjustment / 2

  #Alone
  if current_smd_alone != smdAlone:
      adjustment = smdAlone - current_smd_alone
      all_values[idxAlone] += adjustment / 2
      all_values[idxControl] -= adjustment / 2

  #Set max and min values, and normalize (to ensure positive values)
  min_value = np.min(all_values)
  max_value = np.max(all_values)
  all_values = (all_values - min_value) / (max_value - min_value) * 10

  #add r after normalization to increase level
  all_values[idxControl] += r
  all_values[idxAttention] += r
  all_values[idxTangibles] += r
  all_values[idxEscape] += r
  all_values[idxAlone] += r

  #Combine labels and values in same list
  ME_data = [labels, all_values, {'Attention': smdAttention, 'Tangibles': smdTangibles, 'Escape': smdEscape, 'Alone': smdAlone}]

  #Return ME data
  return(ME_data)



In [ ]:
#Function to produce ME graph

def ATgraph(ME_data, trControl, trAttention, trTangibles, trEscape, trAlone):

    #Identify indices for all conditions
    Control, = np.where(ME_data[0] == 'Control')
    Attention, = np.where(ME_data[0] == 'Attention')
    Tangibles, = np.where(ME_data[0] == 'Tangibles')
    Escape, = np.where(ME_data[0] == 'Escape')
    Alone, = np.where(ME_data[0] == 'Alone')

    #Extract values for each condition
    valuesControl = ME_data[1][Control]
    valuesTest = ME_data[1][Attention]
    valuesTestB = ME_data[1][Tangibles]
    valuesTestC = ME_data[1][Escape]
    valuesTestD = ME_data[1][Alone]

    #Calculate mean and sd of control
    meanControl = np.mean(valuesControl)
    stdControl = np.std(valuesControl)

    #Initialize figure
    fig = plt.figure()
    ax = fig.add_subplot(111)

    #Plot data
    plt.plot(Control+1, valuesControl, 'ks-', label = 'Control')
    plt.plot(Attention+1, valuesTest, 'ko-', label = 'Attention')
    plt.plot(Tangibles+1, valuesTestB, 'go-', label = 'Tangibles')
    plt.plot(Escape+1, valuesTestC, 'ro-', label = 'Escape')
    plt.plot(Alone+1, valuesTestD, 'bo-', label = 'Alone')

    #Add axes titles
    plt.xlabel('Session')
    plt.ylabel('Behavior')

    #Add legend to graph
    ax.legend(loc='upper right', frameon=False)

    #Adjust height of graph
    plt.ylim(0, np.max(ME_data[1]*1.5))

    #Remove right and top borders
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)


In [ ]:
#Run simulations
import math
import pickle
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import gc
import os
import datetime
import random

#Mount GD
from google.colab import drive
drive.mount('/content/drive')

#Foler in GD
run_id = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = f"/content/drive/My Drive/5C_simulation_outputs_{run_id}"
pdf_dir = os.path.join(output_dir, "5Cpdfs")
pickle_dir = os.path.join(output_dir, "5Cpickles")

#Ensuring output directories exist - problem-solved
os.makedirs(pdf_dir, exist_ok=True)
os.makedirs(pickle_dir, exist_ok=True)

#Parameter list
nb_points_list = [6, 10]
a_list = [0, 0.2]
tr_list_Control   = [-30, 0, 0, 0, 30]
tr_list_Attention = [-30, 0, 0, 0, 30]
tr_list_Tangibles = [-30, 0, 0, 0, 30]
tr_list_Escape    = [-30, 0, 0, 0, 30]
tr_list_Alone     = [-30, 0, 0, 0, 30]
smd_list_Attention = [0, 1, 1.44, 2, 3]
smd_list_Tangibles = [0, 1, 1.44, 2, 3]
smd_list_Escape    = [0, 1, 1.44, 2, 3]
smd_list_Alone     = [0, 1, 1.44, 2, 3]
r_list = [0, 2]
ct = 10

#Chunk
chunk_size = 500
simulation_counter = 0
chunk_index = 2107
chunk_data = []
total_simulations_target = 1_000_000

#Pdfs
pdf_path = os.path.join(pdf_dir, "Randomized_ME5graphFinal.pdf")
pp = PdfPages(pdf_path)

#Perform sumulations
def perform_randomized_simulations():
    global simulation_counter, chunk_index, chunk_data
    generated_count = [0, 0, 0]  #Counts for each SMD distribution category (0, 1, 2) - problem-solved to ensure balance

    #Initialize lists
    all_ME_data = []
    true_values = np.empty((0,))
    trend_values = np.empty((0, 5))

    try:
        while sum(generated_count) < total_simulations_target:

            #Randomly select parameters
            nb = random.choice(nb_points_list)
            a = random.choice(a_list)

            #Decision of trend/rate
            if random.random() < 0.625:
                trControl   = 0
                trAttention = 0
                trTangibles = 0
                trEscape    = 0
                trAlone     = 0
                r = 0
            else:
                trControl   = random.choice(tr_list_Control)
                trAttention = random.choice(tr_list_Attention)
                trTangibles = random.choice(tr_list_Tangibles)
                trEscape    = random.choice(tr_list_Escape)
                trAlone     = random.choice(tr_list_Alone)
                r = random.choice(r_list)

            #Category of SMD (undifferentiated, differentiated - singular function, or differentiated - multiple functions)
            desired_category = random.choice([0, 1, 2])
            category = desired_category

            #Define differentiation
            low_pool = [0, 1]
            high_pool = [1.55, 2, 3]
            conditions = ['Attention', 'Tangibles', 'Escape', 'Alone']

            #SMD values
            if desired_category == 0:
                #All conditions low
                smdAttention = random.choice(low_pool)
                smdTangibles = random.choice(low_pool)
                smdEscape = random.choice(low_pool)
                smdAlone = random.choice(low_pool)
            elif desired_category == 1:
                #Exactly one high condition - picked at random (problem-solved)
                high_condition = random.choice(conditions)
                smdAttention = random.choice(high_pool) if high_condition == 'Attention' else random.choice(low_pool)
                smdTangibles = random.choice(high_pool) if high_condition == 'Tangibles' else random.choice(low_pool)
                smdEscape = random.choice(high_pool) if high_condition == 'Escape' else random.choice(low_pool)
                smdAlone = random.choice(high_pool) if high_condition == 'Alone' else random.choice(low_pool)
            elif desired_category == 2:
                #More than one condition high - picked at random (problem-solved)
                num_high = random.randint(2, 4)
                high_conditions = random.sample(conditions, num_high)
                smdAttention = random.choice(high_pool) if 'Attention' in high_conditions else random.choice(low_pool)
                smdTangibles = random.choice(high_pool) if 'Tangibles' in high_conditions else random.choice(low_pool)
                smdEscape = random.choice(high_pool) if 'Escape' in high_conditions else random.choice(low_pool)
                smdAlone = random.choice(high_pool) if 'Alone' in high_conditions else random.choice(low_pool)
                # category remains desired_category

            #Generate data
            ME_data = create_ME_data(
                a=a, trControl=trControl, trAttention=trAttention, trTangibles=trTangibles,
                trEscape=trEscape, trAlone=trAlone, ct=ct, nb_points=nb,
                smdAttention=smdAttention, smdTangibles=smdTangibles,
                smdEscape=smdEscape, smdAlone=smdAlone, r=r, alternation='semi-random'
            )

            #Increase simulation counter
            simulation_counter += 1
            individual_pdf_path = os.path.join(pdf_dir, f"simulation_{simulation_counter:07d}.pdf")

            #Create figure
            plt.figure()
            ATgraph(ME_data, trControl, trAttention, trTangibles, trEscape, trAlone)

            #Save figure
            plt.savefig(individual_pdf_path, format='pdf')

            #Save figure to combined pdf
            pp.savefig()

            #Remove figure from memory
            plt.close()

            #Increase category count
            generated_count[category] += 1

            #Save record
            record = {
                'labels': ME_data[0],
                'datapoint_values': ME_data[1],
                'smd_values': ME_data[2],
                'category': category,
                'trend': [trControl, trAttention, trTangibles, trEscape, trAlone]
            }

            #Append record to chunk
            chunk_data.append(record)

            #Pickle and clear chunk data (problem-solved)
            if len(chunk_data) >= chunk_size:
                chunk_index += 1
                filename = os.path.join(pickle_dir, f'simulation_chunk_{chunk_index:03d}.pkl')
                with open(filename, 'wb') as f:
                    pickle.dump(chunk_data, f)
                print(f'Saved chunk {chunk_index} with {len(chunk_data)} records to {filename}')
                chunk_data = []

    except Exception as e:
        print("An error occurred:", e)
    finally:
        #Save chunk data
        if chunk_data:
            chunk_index += 1
            filename = os.path.join(pickle_dir, f'simulation_chunk_{chunk_index:03d}.pkl')
            with open(filename, 'wb') as f:
                pickle.dump(chunk_data, f)
            print(f'Saved final chunk {chunk_index} with {len(chunk_data)} records to {filename}')

perform_randomized_simulations()

#Close pdf
pp.close()



Mounted at /content/drive


<ipython-input-4-4b115aabf84f>:24: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure()
<ipython-input-5-54be8ab6fc58>:138: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure()


Keeping session alive...
Keeping session alive...
Saved chunk 2108 with 500 records to /content/drive/My Drive/5C_simulation_outputs_2025-02-11_14-43-41/5Cpickles/simulation_chunk_2108.pkl
Keeping session alive...
Saved chunk 2109 with 500 records to /content/drive/My Drive/5C_simulation_outputs_2025-02-11_14-43-41/5Cpickles/simulation_chunk_2109.pkl
Keeping session alive...
Keeping session alive...
Saved chunk 2110 with 500 records to /content/drive/My Drive/5C_simulation_outputs_2025-02-11_14-43-41/5Cpickles/simulation_chunk_2110.pkl
Keeping session alive...
Saved chunk 2111 with 500 records to /content/drive/My Drive/5C_simulation_outputs_2025-02-11_14-43-41/5Cpickles/simulation_chunk_2111.pkl
Keeping session alive...
Keeping session alive...
Saved chunk 2112 with 500 records to /content/drive/My Drive/5C_simulation_outputs_2025-02-11_14-43-41/5Cpickles/simulation_chunk_2112.pkl
Keeping session alive...
Keeping session alive...
Saved chunk 2113 with 500 records to /content/drive/My 